In [ ]:
# | default_exp cli

In [ ]:
# | export
import typer
import sys
from pathlib import Path
from typing import Optional
import pandas as pd
import structlog

log = structlog.get_logger()
app = typer.Typer(name="kreview", help="ctDNA fragmentomics feature evaluation")


def version_callback(value: bool):
    if value:
        import kreview

        typer.echo(f"kreview version: {kreview.__version__}")
        raise typer.Exit()


@app.callback()
def main(
    version: Optional[bool] = typer.Option(
        None, "--version", callback=version_callback, is_eager=True
    ),
):
    """Kreview: ctDNA fragmentomics feature evaluation framework."""
    pass


In [ ]:
# | export
@app.command()
def label(
    cancer_samplesheet: Path = typer.Option(..., help="Cancer samplesheet CSV"),
    healthy_xs1_samplesheet: Path = typer.Option(
        ..., help="Healthy XS1 samplesheet CSV"
    ),
    healthy_xs2_samplesheet: Path = typer.Option(
        ..., help="Healthy XS2 samplesheet CSV"
    ),
    cbioportal_dir: Path = typer.Option(..., help="Directory with cBioPortal files"),
    output: Path = typer.Option("labels.parquet", help="Output parquet file"),
    min_vaf: float = typer.Option(
        0.01, help="Min VAF for Possible ctDNA+ (default 1%)"
    ),
    min_variants: int = typer.Option(
        1, help="Min # variants passing VAF for Possible ctDNA+"
    ),
    ch_hotspot_maf: Path = typer.Option(
        None,
        help="Optional TSV of CH hotspot variants for CH-only demotion. "
        "Samples with only CH mutations are demoted to Possible ctDNA−.",
    ),
):
    """Generate ctDNA labels without feature evaluation."""
    from kreview.core import Paths, LabelConfig
    from kreview.labels import CtDNALabeler

    print("=== kreview label ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --cancer-samplesheet : {cancer_samplesheet}", flush=True)
    print(f"  --healthy-xs1       : {healthy_xs1_samplesheet}", flush=True)
    print(f"  --healthy-xs2       : {healthy_xs2_samplesheet}", flush=True)
    print(f"  --cbioportal-dir    : {cbioportal_dir}", flush=True)
    print(f"  --output            : {output}", flush=True)
    print(f"  --min-vaf           : {min_vaf}", flush=True)
    print(f"  --min-variants      : {min_variants}", flush=True)
    print(f"  --ch-hotspot-maf    : {ch_hotspot_maf or 'disabled'}", flush=True)
    print("", flush=True)

    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        [],
    )
    # Metadata is ~1 row/sample — load everything in a single large batch.
    config = LabelConfig(
        min_vaf=min_vaf,
        min_variants=min_variants,
        chunk_size=15_000,
        ch_hotspot_maf=ch_hotspot_maf,
    )

    labeler = CtDNALabeler(paths, config)
    labels = labeler.label_all()
    labels.to_parquet(output, index=False)
    typer.echo(f"Labels written to {output}")
    typer.echo(labels["label"].value_counts().to_string())


In [ ]:
# | export
@app.command()
def features_list():
    """List all registered feature evaluators."""
    from kreview.registry import get_all_evaluators

    evals = get_all_evaluators()
    for e in evals:
        typer.echo(
            f"Tier {getattr(e, 'tier', '?')}: {getattr(e, 'name', type(e).__name__)} ({getattr(e, 'source_file', '?')})"
        )


In [ ]:
# | export
import numpy as np
import time


def _impute(df, strategy: str):
    """Apply imputation strategy to a feature DataFrame.

    Args:
        df: DataFrame with numeric features (may contain NaN).
        strategy: One of 'zero', 'mean', 'median'.

    Returns:
        DataFrame with NaN values filled according to strategy.
    """
    if strategy == "mean":
        return df.fillna(df.mean())
    elif strategy == "median":
        return df.fillna(df.median())
    elif strategy != "zero":
        import structlog

        structlog.get_logger().warning(
            "unknown_impute_strategy",
            strategy=strategy,
            fallback="zero",
        )
    return df.fillna(0)  # default: zero


def _extract_evaluator(
    evaluator,
    paths,
    all_sample_ids: list[str],
    chunk_size: int,
    out_path: Path,
    labels_df: pd.DataFrame,
    *,
    _echo=print,
) -> pd.DataFrame | None:
    """Extract features for a single evaluator and write the matrix parquet.

    Tries SQL pushdown first (Path A); falls back to chunked Python
    streaming (Path B) when the evaluator doesn't support SQL or SQL
    returns empty results.

    Returns:
        Merged DataFrame (labels + features) written to disk, or None
        if extraction produced no data.
    """
    from kreview.core import iter_feature_chunks, run_feature_sql

    _echo(f"  Extracting '{evaluator.name}'...")
    t1 = time.time()

    # --- Path A: SQL Pushdown (if evaluator supports it) ---
    sql_query = evaluator.extract_sql()
    feat_matrix = None

    if sql_query is not None:
        _echo(f"  Using SQL pushdown for '{evaluator.name}'...")
        sql_df = run_feature_sql(
            sql_query,
            str(evaluator.source_file),
            paths.krewlyzer_dirs,
            set(all_sample_ids),
        )
        if not sql_df.empty:
            feat_matrix = sql_df
            extract_sec = time.time() - t1
            n_samples = (
                sql_df["sample_id"].nunique()
                if "sample_id" in sql_df.columns
                else len(sql_df)
            )
            _echo(
                f"  SQL pushdown complete: {n_samples} samples "
                f"in {extract_sec:.1f}s"
            )
            # Normalize sample_id column name for downstream merge
            if "sample_id" in sql_df.columns and "SAMPLE_ID" not in sql_df.columns:
                feat_matrix = feat_matrix.rename(columns={"sample_id": "SAMPLE_ID"})
        else:
            _echo(
                f"  SQL pushdown returned empty result for '{evaluator.name}', "
                f"falling back to chunked extraction..."
            )
            sql_query = None  # Force fallback to Path B

    # --- Path B: Chunked Python streaming (default fallback) ---
    if sql_query is None:
        _echo(f"  Streaming cohort from DuckDB (chunk_size={chunk_size})...")
        partial_results = []
        failed_samples = []
        total_samples_processed = 0
        total_rows_read = 0

        for chunk_df, chunk_idx, n_chunks in iter_feature_chunks(
            str(evaluator.source_file),
            paths.krewlyzer_dirs,
            set(all_sample_ids),
            chunk_size=chunk_size,
        ):
            sid_col = "sample_id" if "sample_id" in chunk_df.columns else "SAMPLE_ID"
            chunk_sample_ids = chunk_df[sid_col].unique()
            chunk_n_rows = len(chunk_df)
            total_rows_read += chunk_n_rows

            _echo(
                f"  Chunk {chunk_idx + 1}/{n_chunks}: "
                f"{len(chunk_sample_ids)} samples, {chunk_n_rows:,} rows"
            )

            # Extract features from each sample within this chunk
            chunk_extracted = 0
            for sample_id in chunk_sample_ids:
                sample_df = chunk_df[chunk_df[sid_col] == sample_id]
                try:
                    res = evaluator.extract(sample_df)
                    if res:
                        res["SAMPLE_ID"] = sample_id
                        partial_results.append(pd.DataFrame([res]))
                        chunk_extracted += 1
                except Exception as exc:
                    log.warning(
                        "sample_extraction_failed",
                        evaluator=evaluator.name,
                        sample_id=sample_id,
                        error=str(exc),
                    )
                    failed_samples.append(sample_id)

            total_samples_processed += len(chunk_sample_ids)
            _echo(
                f"    Extracted {chunk_extracted}/" f"{len(chunk_sample_ids)} samples"
            )

            # Explicit memory release — reclaim ~5GB before next chunk
            del chunk_df

        if total_samples_processed == 0:
            _echo(f"  WARNING: No data found for {evaluator.name}, skipping")
            return None

        extract_sec = time.time() - t1
        _echo(
            f"  Extraction complete: {total_samples_processed} samples, "
            f"{total_rows_read:,} rows read in {extract_sec:.1f}s"
        )

        if failed_samples:
            failed_path = out_path / f"{evaluator.name}_failed_samples.csv"
            pd.DataFrame({"SAMPLE_ID": failed_samples}).to_csv(failed_path, index=False)
            _echo(
                f"  WARNING: {len(failed_samples)} samples failed extraction. "
                f"Saved to {failed_path.name}"
            )
            log.warning(
                "extraction_failures",
                evaluator=evaluator.name,
                n_failed=len(failed_samples),
                output=str(failed_path),
            )

        if not partial_results:
            _echo(f"  WARNING: No results for {evaluator.name}, skipping")
            return None

        feat_matrix = pd.concat(partial_results, ignore_index=True)

    # Guard: both Path A and Path B should set feat_matrix before reaching
    # this point (or return None). This is a safety net.
    if feat_matrix is None:
        _echo(f"  WARNING: No feature matrix produced for {evaluator.name}, skipping")
        return None

    merged = pd.merge(labels_df, feat_matrix, on="SAMPLE_ID", how="inner")
    out_p = out_path / f"{evaluator.name}_matrix.parquet"
    merged.to_parquet(out_p, index=False)
    _echo(f"  Matrix: {merged.shape[0]} samples x {merged.shape[1]} cols -> {out_p}")

    return merged


def _find_quarto() -> str:
    """Discover the quarto binary from PATH or well-known install locations."""
    import shutil

    # 1. Check PATH first
    found = shutil.which("quarto")
    if found:
        return found

    # 2. Check well-known install locations
    candidates = [
        # macOS Positron bundled Quarto
        "/Applications/Positron.app/Contents/Resources/app/quarto/bin/quarto",
        # macOS standalone Quarto install
        "/Applications/quarto/bin/quarto",
        # Homebrew (Apple Silicon)
        "/opt/homebrew/bin/quarto",
        # Homebrew (Intel)
        "/usr/local/bin/quarto",
        # Linux system install
        "/usr/bin/quarto",
        # User-local install
        str(Path.home() / ".local" / "bin" / "quarto"),
        # Conda env
        str(Path(sys.executable).parent / "quarto"),
    ]
    for c in candidates:
        if Path(c).is_file():
            return c

    return "quarto"  # Fall back to bare name (will raise FileNotFoundError)


def _render_quarto_report(
    matrix_path: str,
    feat_name: str,
    report_dir: Path,
    python_exe: str,
    cvd_safe: bool = False,
    shap_samples: int = 500,
    shap_features: int = 10,
) -> tuple[bool, str]:
    """Render a Quarto HTML dashboard for a single feature matrix."""
    import subprocess
    import os

    pkg_dir = Path(__file__).resolve().parent
    template_dir = pkg_dir / "templates"
    template_file = template_dir / "report_template.qmd"

    if not template_file.exists():
        return False, f"Template not found at {template_file}"

    env = os.environ.copy()
    env["QUARTO_PYTHON"] = str(python_exe)

    quarto_bin = _find_quarto()
    out_html = Path(report_dir) / f"{feat_name}_dashboard.html"
    log_file = Path(report_dir) / f"{feat_name}_render.log"
    cmd = [
        quarto_bin,
        "render",
        "report_template.qmd",
        "--log-level",
        "DEBUG",
        "-P",
        f"matrix_path:{Path(matrix_path).absolute()}",
        "-P",
        f"feature_name:{feat_name}",
        "-P",
        f"cvd_safe:{str(cvd_safe).lower()}",
        "-P",
        f"shap_samples:{shap_samples}",
        "-P",
        f"shap_features:{shap_features}",
        "--output",
        out_html.name,
        "--output-dir",
        str(out_html.parent.absolute()),
    ]

    try:
        subprocess.run(
            cmd,
            env=env,
            capture_output=True,
            text=True,
            check=True,
            timeout=600,
            cwd=str(template_dir),
        )
        return True, str(out_html)
    except subprocess.CalledProcessError as exc:
        # Write full output to disk for debugging — truncated console output hides errors
        with open(log_file, "w") as f:
            f.write(f"=== QUARTO RENDER DEBUG LOG: {feat_name} ===\n")
            f.write(f"CMD: {' '.join(cmd)}\n")
            f.write(f"EXIT CODE: {exc.returncode}\n\n")
            f.write("=== STDOUT ===\n")
            f.write(exc.stdout or "(empty)")
            f.write("\n\n=== STDERR ===\n")
            f.write(exc.stderr or "(empty)")
        # Return concise error + pointer to full log
        err_summary = f"Exit code {exc.returncode}. Full log: {log_file}\n"
        # Extract actual error lines (skip cell progress noise)
        for stream in [exc.stdout, exc.stderr]:
            if not stream:
                continue
            for line in stream.splitlines():
                line_s = line.strip()
                if any(
                    kw in line_s.lower()
                    for kw in [
                        "error",
                        "exception",
                        "traceback",
                        "failed",
                        "fatal",
                        "not found",
                    ]
                ):
                    err_summary += f"  >> {line_s}\n"
        return False, err_summary
    except subprocess.TimeoutExpired:
        return False, "Timeout (>600s)"


@app.command()
def run(
    cancer_samplesheet: Path = typer.Option(...),
    healthy_xs1_samplesheet: Path = typer.Option(...),
    healthy_xs2_samplesheet: Path = typer.Option(...),
    cbioportal_dir: Path = typer.Option(...),
    krewlyzer_dir: list[str] = typer.Option(..., help="krewlyzer output directory"),
    output: Path = typer.Option("output/", help="Output directory"),
    min_vaf: float = typer.Option(0.01),
    min_fragments: int = typer.Option(2000),
    min_variants: int = typer.Option(1),
    features: Optional[str] = typer.Option(
        None, help="Comma-separated features to run"
    ),
    tier: Optional[int] = typer.Option(None, help="Run features of this tier only"),
    cvd_safe: bool = typer.Option(
        False,
        "--cvd-safe",
        help="Render dashboards and plots using an Okabe-Ito Colorblind-Safe palette instead of default neon.",
    ),
    skip_report: bool = typer.Option(False, help="Skip HTML report generation"),
    cv_folds: int = typer.Option(
        5, "--cv-folds", help="Number of cross-validation folds (3-20, default 5)"
    ),
    impute_strategy: str = typer.Option(
        "median",
        "--impute-strategy",
        help="Imputation strategy for missing values: median, mean, or zero",
    ),
    export_duckdb: bool = typer.Option(
        False,
        "--export-duckdb",
        help="Export a persistent duckdb data lake containing all feature matrices",
    ),
    chunk_size: str = typer.Option(
        "auto",
        "--chunk-size",
        help=(
            "Samples per DuckDB read batch. 'auto' (default) probes parquet "
            "row density at runtime, or pass an integer to override "
            "(e.g. --chunk-size 200)."
        ),
    ),
    top_n: int = typer.Option(
        None,
        "--top-n",
        help="[DEPRECATED] Use --top-percentile instead. If set, overrides --top-percentile with a fixed count.",
    ),
    top_percentile: float = typer.Option(
        10.0,
        "--top-percentile",
        help="Top X%% of features to select per metric (AUC, MI). The union of both sets feeds into models.",
    ),
    shap_samples: int = typer.Option(
        500,
        "--shap-samples",
        help="Max samples for SHAP explainability computation in dashboards. Lower values reduce memory usage.",
    ),
    shap_features: int = typer.Option(
        10,
        "--shap-features",
        help="Max features to visualize in SHAP beeswarm/waterfall plots.",
    ),
    resume: bool = typer.Option(
        False,
        "--resume",
        help="Skip evaluators whose model results already exist in the output directory.",
    ),
    compute_univariate_auc: bool = typer.Option(
        True,
        "--compute-univariate-auc",
        help="Compute per-feature univariate LR AUC. Required for hybrid selection (default: True).",
    ),
    ch_hotspot_maf: Path = typer.Option(
        None,
        "--ch-hotspot-maf",
        help="Optional TSV of CH hotspot variants for CH-only demotion. "
        "Samples with only CH mutations are demoted to Possible ctDNA−.",
    ),
):
    """Run full pipeline: label → extract → evaluate → report."""
    from kreview.core import Paths, LabelConfig
    from kreview.labels import CtDNALabeler
    from kreview.registry import get_all_evaluators

    from kreview.eval_engine import evaluate_feature, single_feature_model
    import glob
    import json

    def _echo(msg):
        print(msg, flush=True)

    if cv_folds < 3 or cv_folds > 20:
        _echo("ERROR: --cv-folds must be between 3 and 20")
        raise typer.Exit(code=1)
    if impute_strategy not in ["median", "mean", "zero"]:
        _echo("ERROR: --impute-strategy must be median, mean, or zero")
        raise typer.Exit(code=1)
    if not (1.0 <= top_percentile <= 100.0):
        _echo("ERROR: --top-percentile must be between 1.0 and 100.0")
        raise typer.Exit(code=1)
    if top_n is not None:
        _echo(
            "WARNING: --top-n is deprecated and will be removed in v0.1.0. "
            "Use --top-percentile instead. Ignoring --top-n in favor of --top-percentile."
        )

    _echo("=== kreview run ===")
    _echo("Configuration:")
    _echo(f"  --cancer-samplesheet : {cancer_samplesheet}")
    _echo(f"  --healthy-xs1       : {healthy_xs1_samplesheet}")
    _echo(f"  --healthy-xs2       : {healthy_xs2_samplesheet}")
    _echo(f"  --cbioportal-dir    : {cbioportal_dir}")
    _echo(f"  --krewlyzer-dir     : {krewlyzer_dir}")
    _echo(f"  --output            : {output}")
    _echo(f"  --min-vaf           : {min_vaf}")
    _echo(f"  --min-variants      : {min_variants}")
    _echo(f"  --features          : {features or 'ALL'}")
    _echo(f"  --tier              : {tier or 'ALL'}")
    _echo(f"  --cv-folds          : {cv_folds}")
    _echo(f"  --top-percentile    : {top_percentile}")
    _echo(f"  --impute-strategy   : {impute_strategy}")
    _echo(f"  --chunk-size        : {chunk_size}")
    _echo(f"  --shap-samples      : {shap_samples}")
    _echo(f"  --shap-features     : {shap_features}")
    _echo(f"  --skip-report       : {skip_report}")
    _echo(f"  --cvd-safe          : {cvd_safe}")
    _echo(f"  --export-duckdb     : {export_duckdb}")
    _echo(f"  --resume            : {resume}")
    _echo(f"  --compute-univariate-auc : {compute_univariate_auc}")
    _echo(f"  --ch-hotspot-maf    : {ch_hotspot_maf or 'disabled'}")
    _echo("")

    # ── Validate --chunk-size ──
    # 'auto' passes through to iter_feature_chunks which probes row density.
    # An integer string is converted; anything else is rejected early.
    effective_chunk: int | str = chunk_size
    if chunk_size != "auto":
        try:
            effective_chunk = int(chunk_size)
            if effective_chunk < 1:
                raise ValueError("chunk_size must be positive")
        except ValueError:
            _echo(
                f"ERROR: --chunk-size must be 'auto' or a positive integer, "
                f"got '{chunk_size}'"
            )
            raise typer.Exit(code=1)

    # ── Step 1: Labels ──
    _echo("Step 1: Generating Labels...")
    t0 = time.time()
    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        list(krewlyzer_dir),
    )
    # LabelConfig uses a fixed chunk_size — metadata is always ~1 row/sample.
    config = LabelConfig(
        min_vaf=min_vaf,
        min_variants=min_variants,
        chunk_size=15_000,
        ch_hotspot_maf=ch_hotspot_maf,
    )
    labeler = CtDNALabeler(paths, config)
    labels_df = labeler.label_all()
    _echo(f"  Labels: {len(labels_df)} samples in {time.time()-t0:.1f}s")
    _echo(f"  Distribution:\n{labels_df['label'].value_counts().to_string()}")

    # ── Step 2: Resolve evaluators ──
    _echo("Step 2: Resolving Feature Evaluators...")
    all_evals = get_all_evaluators()
    target_evals = []
    feat_filter = features.split(",") if features else []
    for e in all_evals:
        if tier is not None and getattr(e, "tier", -1) != tier:
            continue
        if feat_filter and getattr(e, "name", "") not in feat_filter:
            continue
        target_evals.append(e)

    if not target_evals:
        _echo(f"ERROR: No evaluators matched. Available: {[e.name for e in all_evals]}")
        raise typer.Exit(code=1)
    _echo(f"  Matched: {[e.name for e in target_evals]}")

    out_path = Path(output)
    out_path.mkdir(parents=True, exist_ok=True)
    all_sample_ids = list(labels_df["SAMPLE_ID"].unique())

    for e in target_evals:
        _echo(f"\n{'='*60}")
        _echo(f"Processing evaluator: {e.name}")

        # ── Resume checkpoint: skip if model results already exist ──
        # GAP-4: Also check that the JSON contains OOF predictions (added in
        # v0.8.4). JSONs from older runs lack oof_labels and will produce
        # legacy (leaky) ROC plots. Warn but still skip to avoid re-running
        # expensive evaluations — user can delete the JSON to force recompute.
        if resume:
            checkpoint = out_path / f"{e.name}_model_results.json"
            if checkpoint.exists():
                try:
                    import json as _json

                    with open(checkpoint) as _f:
                        _chk = _json.load(_f)
                    if "oof_labels" not in _chk:
                        _echo(
                            f"  SKIP (--resume): {checkpoint.name} exists but MISSING oof_labels. "
                            f"Delete this file and re-run to get OOF-based ROC plots."
                        )
                    elif "selection_qc" not in _chk:
                        _echo(
                            f"  SKIP (--resume): {checkpoint.name} uses legacy Cohen's D selection. "
                            f"Delete to re-run with hybrid union feature selection."
                        )
                    else:
                        _echo(f"  SKIP (--resume): {checkpoint.name} already exists")
                except Exception:
                    _echo(
                        f"  SKIP (--resume): {checkpoint.name} exists (could not verify contents)"
                    )
                continue

        # ── Step 3: Extract features ──
        # Delegates to _extract_evaluator() which handles both SQL pushdown
        # (Path A) and chunked Python streaming (Path B).
        merged = _extract_evaluator(
            e,
            paths,
            all_sample_ids,
            effective_chunk,
            out_path,
            labels_df,
            _echo=_echo,
        )
        if merged is None:
            continue

        # ── Step 4: Statistical Evaluation ──
        _echo(f"Step 4: Running statistical evaluation for '{e.name}'...")
        t3 = time.time()

        # Identify numeric feature columns (exclude label/metadata cols).
        # Uses the canonical LABEL_META_COLS set from core — single source of truth.
        from kreview.core import LABEL_META_COLS

        numeric_cols = [
            c
            for c in merged.select_dtypes(include=np.number).columns
            if c not in LABEL_META_COLS
        ]
        _echo(f"  Found {len(numeric_cols)} numeric feature columns")

        eval_results = []
        for col in numeric_cols:
            res = evaluate_feature(
                merged[col],
                merged["label"],
                total_fragments=merged.get("n_total_somatic_snvs"),
                max_vaf=merged.get("max_vaf"),
            )
            res["feature_column"] = col
            eval_results.append(res)

        eval_df = pd.DataFrame(eval_results)

        # ── Feature Scoring: Univariate AUC + Mutual Information ──
        # Build binary target once for both scoring methods.
        # This target matches the downstream model target (positives vs negatives).
        _model_mask = merged["label"].isin(
            [
                "True ctDNA+",
                "Possible ctDNA+",
                "Healthy Normal",
                "Possible ctDNA-",
                "Possible ctDNA\u2212",
            ]
        )
        _m_df = merged[_model_mask]
        _y_scoring = (
            _m_df["label"].isin(["True ctDNA+", "Possible ctDNA+"]).astype(int).values
        )

        if compute_univariate_auc:
            from kreview.eval_engine import univariate_auc as _uauc

            _echo(f"  Computing univariate AUC for {len(numeric_cols)} features...")
            uauc_scores = []
            for col in numeric_cols:
                auc_val = _uauc(_m_df[col], _y_scoring, n_folds=cv_folds)
                uauc_scores.append(auc_val)
            eval_df["univariate_auc"] = uauc_scores
        else:
            _echo(
                "  WARNING: Univariate AUC disabled (--no-compute-univariate-auc). "
                "Feature selection will use mutual information only."
            )
            log.warning(
                "univariate_auc_disabled",
                evaluator=e.name,
                impact="selection_mi_only",
            )

        # Always compute mutual information (fast, no CV needed)
        from kreview.eval_engine import mutual_info_score as _mi_score

        _echo(f"  Computing mutual information for {len(numeric_cols)} features...")
        mi_scores = []
        for col in numeric_cols:
            mi_val = _mi_score(_m_df[col], _y_scoring)
            mi_scores.append(mi_val)
        eval_df["mutual_info"] = mi_scores

        log.info(
            "feature_scoring_complete",
            evaluator=e.name,
            n_features=len(numeric_cols),
            n_auc_above_055=(
                int((eval_df["univariate_auc"] > 0.55).sum())
                if "univariate_auc" in eval_df.columns
                else 0
            ),
            n_mi_above_zero=int((eval_df["mutual_info"] > 0.0).sum()),
        )

        eval_out = out_path / f"{e.name}_eval_stats.parquet"
        eval_df.to_parquet(eval_out, index=False)
        _echo(f"  Eval stats: {eval_df.shape[0]} features -> {eval_out}")

        # Reuse the scoring target for modeling (same 4-tier label mask).
        # .copy() ensures downstream mutations don't affect the scoring DataFrame.
        model_df = _m_df.copy()
        y = _y_scoring
        if len(model_df) >= 20 and len(np.unique(y)) == 2:
            # ── Hybrid Union Feature Selection ──
            # Select top X% by Univariate AUC ∪ top X% by Mutual Information.
            # This captures both linear (AUC) and non-linear (MI) predictors,
            # ensuring the downstream models receive a diverse, high-quality
            # feature set regardless of evaluator dimensionality.
            n_keep = max(1, int(len(numeric_cols) * (top_percentile / 100.0)))

            top_by_auc = set()
            top_by_mi = set()

            if "univariate_auc" in eval_df.columns:
                top_by_auc = set(
                    eval_df.nlargest(n_keep, "univariate_auc")["feature_column"]
                )
            if "mutual_info" in eval_df.columns:
                top_by_mi = set(
                    eval_df.nlargest(n_keep, "mutual_info")["feature_column"]
                )

            # Union: keep features that are strong in either metric
            union_feats = list(top_by_auc | top_by_mi)

            # Fallback: if both metrics are empty, take first n_keep features
            if not union_feats:
                log.warning(
                    "selection_fallback",
                    evaluator=e.name,
                    reason="no_scored_features",
                )
                union_feats = numeric_cols[:n_keep]

            top_feats = union_feats

            # Build selection QC metadata for JSON output and reports
            selection_qc = {
                "method": "hybrid_union",
                "total_input_features": len(numeric_cols),
                "target_percentile": top_percentile,
                "n_keep_per_metric": n_keep,
                "n_selected_union": len(top_feats),
                "n_overlap_both": len(top_by_auc & top_by_mi),
                "n_auc_only": len(top_by_auc - top_by_mi),
                "n_mi_only": len(top_by_mi - top_by_auc),
            }

            log.info(
                "feature_selection_complete",
                evaluator=e.name,
                **selection_qc,
            )
            _echo(
                f"  Feature Selection (top {top_percentile}%): "
                f"{len(top_feats)}/{len(numeric_cols)} features "
                f"(AUC∩MI={selection_qc['n_overlap_both']}, "
                f"AUC-only={selection_qc['n_auc_only']}, "
                f"MI-only={selection_qc['n_mi_only']})"
            )

            if top_feats:
                # Variance guard: drop features that are constant across all model samples.
                # Constant features produce AUC=0.500 and waste compute.
                X_imputed = _impute(model_df[top_feats], impute_strategy)
                nonconst = [c for c in top_feats if X_imputed[c].std() > 0]
                n_dropped = len(top_feats) - len(nonconst)
                if n_dropped > 0:
                    _echo(
                        f"  WARNING: Dropped {n_dropped} constant features (zero variance)"
                    )
                    log.info(
                        "variance_guard_dropped",
                        evaluator=e.name,
                        n_dropped=n_dropped,
                        n_remaining=len(nonconst),
                    )
                    top_feats = nonconst

                if not top_feats:
                    _echo(
                        f"  WARNING: All features are constant for {e.name}, skipping model"
                    )
                    continue

                import warnings

                warnings.filterwarnings("ignore", message=".*use_label_encoder.*")

                _echo(f"  Imputation: {impute_strategy}")

                # Reuse cached imputed data (re-slice if variance guard dropped columns)
                X = X_imputed[top_feats].values
                c_types = model_df.get("CANCER_TYPE", None)
                if c_types is not None:
                    c_types = c_types.values
                a_types = model_df.get("access_version", None)
                if a_types is not None:
                    a_types = a_types.values

                model_res, lr_model, rf_model, xgb_model = single_feature_model(
                    X,
                    y,
                    feature_names=top_feats,
                    cancer_types=c_types,
                    assays=a_types,
                    n_folds=cv_folds,
                )
                model_out = out_path / f"{e.name}_model_results.json"

                if "error" not in model_res:
                    model_res["top_features"] = top_feats
                    model_res["selection_qc"] = selection_qc

                import joblib

                # Save models for downstream SHAP / Dashboards
                if lr_model is not None:
                    joblib_out = out_path / f"{e.name}_lr_model.joblib"
                    joblib.dump(lr_model, joblib_out)
                if rf_model is not None:
                    joblib_out = out_path / f"{e.name}_rf_model.joblib"
                    joblib.dump(rf_model, joblib_out)
                if xgb_model is not None:
                    joblib_out = out_path / f"{e.name}_xgb_model.joblib"
                    joblib.dump(xgb_model, joblib_out)

                with open(model_out, "w") as f:
                    json.dump(model_res, f, indent=2, default=str)

                def _fmt(v):
                    return "N/A" if v is None else f"{v:.3f}"

                _echo(
                    f"  Model: AUC_LR={_fmt(model_res.get('auc_lr'))}, "
                    f"AUC_RF={_fmt(model_res.get('auc_rf'))}, "
                    f"AUC_XGB={_fmt(model_res.get('auc_xgb'))} -> {model_out}"
                )

        else:
            n_classes = len(np.unique(y)) if len(y) > 0 else 0
            _echo(
                f"  SKIP model: {e.name} — insufficient data "
                f"(n_samples={len(model_df)}, n_classes={n_classes}, "
                f"need ≥20 samples and 2 classes)"
            )
            log.warning(
                "model_skip_insufficient_data",
                evaluator=e.name,
                n_samples=len(model_df),
                n_classes=n_classes,
            )

        eval_sec = time.time() - t3
        _echo(f"  Evaluation complete in {eval_sec:.1f}s")

    # ── Step 4b: Build Cross-Evaluator Scoreboard ──
    from kreview.scoreboard import build_scoreboard

    sb = build_scoreboard(out_path)
    if not sb.empty:
        sb.to_parquet(out_path / "scoreboard_combined__all.parquet")
        sb.to_csv(out_path / "scoreboard_combined__all.csv", index=False)
        _echo(f"\n  Scoreboard: {len(sb)} evaluators ranked")
        _echo("  Top 5 by AUC:")
        for _, row in sb.head(5).iterrows():
            best = row["best_auc"]
            auc_str = f"{best:.3f}" if not pd.isna(best) else "N/A"
            _echo(f"    {row['evaluator']:<30} AUC={auc_str}")
        _echo("  -> scoreboard_combined__all.parquet")

    # ── Step 5: Generate HTML Dashboards ──
    if not skip_report:
        report_dir = out_path / "reports"
        report_dir.mkdir(parents=True, exist_ok=True)

        matrices = glob.glob(str(out_path / "*_matrix.parquet"))
        if matrices:
            _echo(f"\nStep 5: Generating {len(matrices)} HTML Dashboard(s)...")
            for p in matrices:
                feat_name = Path(p).name.replace("_matrix.parquet", "")
                _echo(f"  Rendering {feat_name}...")
                ok, msg = _render_quarto_report(
                    p,
                    feat_name,
                    report_dir,
                    sys.executable,
                    cvd_safe=cvd_safe,
                    shap_samples=shap_samples,
                    shap_features=shap_features,
                )
                if ok:
                    _echo(f"  Saved: {msg}")
                else:
                    _echo(f"  Quarto FAILED for {feat_name}: {msg}")

    # ── Step 6: DuckLake Export ──
    if export_duckdb:
        _echo("\nStep 6: Exporting Unified DuckLake...")
        import duckdb

        db_path = out_path / "kreview_lake.duckdb"
        if db_path.exists():
            db_path.unlink()
        try:
            con = duckdb.connect(str(db_path))
            matrices = glob.glob(str(out_path / "*_matrix.parquet"))
            for p in matrices:
                table_name = (
                    Path(p).name.replace("_matrix.parquet", "").replace(".", "_")
                )
                _echo(f"  Importing {table_name} into DuckLake...")
                con.execute(
                    f"CREATE TABLE {table_name} AS SELECT * FROM read_parquet('{p}')"
                )
            con.close()
            _echo(f"  DuckLake saved securely directly to: {db_path}")
        except Exception as e:
            _echo(f"  DuckLake creation failed: {e}")

    total_sec = time.time() - t0
    _echo(f"\n=== Workflow complete in {total_sec:.1f}s ===")


In [ ]:
# | export
@app.command()
def report(
    input_dir: Path = typer.Option(..., help="Directory with *_matrix.parquet files"),
    out_dir: Path = typer.Option(
        "reports/", help="Directory to deposit Quarto reports"
    ),
    cvd_safe: bool = typer.Option(
        False,
        "--cvd-safe",
        help="Render dashboards and plots using an Okabe-Ito Colorblind-Safe palette instead of default neon.",
    ),
    shap_samples: int = typer.Option(
        500,
        "--shap-samples",
        help="Max samples for SHAP explainability computation in dashboards.",
    ),
    shap_features: int = typer.Option(
        10,
        "--shap-features",
        help="Max features to visualize in SHAP plots.",
    ),
):
    """Re-generate HTML Dashboards from existing matrix parquet files.

    Scans ``input_dir`` for ``*_matrix.parquet`` files, renders each as a
    standalone Quarto HTML dashboard, and writes them to ``out_dir``.
    Each evaluator is rendered independently so a single failure does not
    block the remaining dashboards.
    """
    import glob
    import sys
    import time as _time

    import structlog

    _log = structlog.get_logger()

    print("=== kreview report ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --input-dir     : {input_dir}", flush=True)
    print(f"  --out-dir       : {out_dir}", flush=True)
    print(f"  --cvd-safe      : {cvd_safe}", flush=True)
    print(f"  --shap-samples  : {shap_samples}", flush=True)
    print(f"  --shap-features : {shap_features}", flush=True)
    print("", flush=True)

    in_path = Path(input_dir).absolute()
    matrices = sorted(glob.glob(str(in_path / "*_matrix.parquet")))
    if not matrices:
        _log.warning("report_no_matrices", input_dir=str(in_path))
        print(f"No *_matrix.parquet files found in {in_path}", flush=True)
        return

    _log.info("report_started", input_dir=str(in_path), n_matrices=len(matrices))

    out_path = Path(out_dir).absolute()
    out_path.mkdir(parents=True, exist_ok=True)

    succeeded = 0
    failed = 0
    failed_names = []

    for p in matrices:
        feat_name = Path(p).name.replace("_matrix.parquet", "")
        print(f"Rendering dashboard for {feat_name}...", flush=True)
        t_start = _time.perf_counter()

        # S-01: Per-evaluator try/except — one failure does not block others
        try:
            ok, msg = _render_quarto_report(
                p,
                feat_name,
                out_path,
                sys.executable,
                cvd_safe=cvd_safe,
                shap_samples=shap_samples,
                shap_features=shap_features,
            )
            elapsed = _time.perf_counter() - t_start
            if ok:
                print(f"  Saved: {msg} ({elapsed:.1f}s)", flush=True)
                _log.info(
                    "report_rendered",
                    evaluator=feat_name,
                    seconds=f"{elapsed:.1f}",
                    output=msg,
                )
                succeeded += 1
            else:
                print(f"  FAILED: {msg} ({elapsed:.1f}s)", flush=True)
                _log.error(
                    "report_render_failed",
                    evaluator=feat_name,
                    seconds=f"{elapsed:.1f}",
                    error=msg[:500],
                )
                failed += 1
                failed_names.append(feat_name)
        except Exception as exc:
            elapsed = _time.perf_counter() - t_start
            print(
                f"  EXCEPTION rendering {feat_name}: {exc} ({elapsed:.1f}s)",
                flush=True,
            )
            _log.error(
                "report_render_exception",
                evaluator=feat_name,
                seconds=f"{elapsed:.1f}",
                error=str(exc),
            )
            failed += 1
            failed_names.append(feat_name)

    # L-03: Summary with pass/fail counts
    total = len(matrices)
    print(
        f"\nReport summary: {succeeded}/{total} succeeded, {failed}/{total} failed",
        flush=True,
    )
    if failed_names:
        print(f"  Failed evaluators: {', '.join(failed_names)}", flush=True)
    _log.info(
        "report_completed",
        succeeded=succeeded,
        failed=failed,
        total=total,
        failed_evaluators=failed_names,
    )


In [ ]:
# | test
# Smoke test
assert hasattr(app, "registered_commands")

In [ ]:
#| export
@app.command()
def extract(
    cancer_samplesheet: Path = typer.Option(..., help="Cancer samplesheet CSV"),
    healthy_xs1_samplesheet: Path = typer.Option(
        ..., help="Healthy XS1 samplesheet CSV"
    ),
    healthy_xs2_samplesheet: Path = typer.Option(
        ..., help="Healthy XS2 samplesheet CSV"
    ),
    cbioportal_dir: Path = typer.Option(..., help="Directory with cBioPortal files"),
    krewlyzer_dir: list[str] = typer.Option(..., help="krewlyzer output directory"),
    output: Path = typer.Option("output/", help="Output directory for matrices"),
    min_vaf: float = typer.Option(
        0.01, help="Min VAF for Possible ctDNA+ (default 1%)"
    ),
    min_variants: int = typer.Option(
        1, help="Min # variants passing VAF for Possible ctDNA+"
    ),
    ch_hotspot_maf: Path = typer.Option(
        None,
        help="Optional TSV of CH hotspot variants for CH-only demotion.",
    ),
    features: Optional[str] = typer.Option(
        None, help="Comma-separated evaluator names (default: all)"
    ),
    tier: Optional[int] = typer.Option(None, help="Run only this tier"),
    chunk_size: int = typer.Option(5000, help="Rows per DuckDB chunk"),
):
    """Label samples and extract feature matrices (no eval/model/report).

    Runs the labeling pipeline, then extracts features for each matched
    evaluator into ``*_matrix.parquet`` files. This is the first half of
    ``kreview run``, designed for parallelized Nextflow execution.
    """
    from kreview.core import Paths, LabelConfig, _calculate_dynamic_chunk_size
    from kreview.labels import CtDNALabeler
    from kreview.evaluators import get_all_evaluators

    print("=== kreview extract ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --cancer-samplesheet : {cancer_samplesheet}", flush=True)
    print(f"  --healthy-xs1       : {healthy_xs1_samplesheet}", flush=True)
    print(f"  --healthy-xs2       : {healthy_xs2_samplesheet}", flush=True)
    print(f"  --cbioportal-dir    : {cbioportal_dir}", flush=True)
    print(f"  --krewlyzer-dir     : {krewlyzer_dir}", flush=True)
    print(f"  --output            : {output}", flush=True)
    print(f"  --features          : {features or 'all'}", flush=True)
    print(f"  --tier              : {tier or 'all'}", flush=True)
    print(f"  --chunk-size        : {chunk_size}", flush=True)
    print(f"  --ch-hotspot-maf    : {ch_hotspot_maf or 'disabled'}", flush=True)
    print("", flush=True)

    t0 = time.time()

    # ── Step 1: Label ──
    print("Step 1: Labeling...", flush=True)
    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        krewlyzer_dir,
    )
    config = LabelConfig(
        min_vaf=min_vaf,
        min_variants=min_variants,
        chunk_size=15_000,
        ch_hotspot_maf=ch_hotspot_maf,
    )
    labeler = CtDNALabeler(paths, config)
    labels_df = labeler.label_all()
    print(f"  Labels: {len(labels_df)} samples in {time.time()-t0:.1f}s", flush=True)
    print(
        f"  Distribution:\n{labels_df['label'].value_counts().to_string()}", flush=True
    )

    # ── Step 2: Resolve evaluators ──
    print("Step 2: Resolving Feature Evaluators...", flush=True)
    all_evals = get_all_evaluators()
    target_evals = []
    feat_filter = features.split(",") if features else []
    for e in all_evals:
        if tier is not None and getattr(e, "tier", -1) != tier:
            continue
        if feat_filter and getattr(e, "name", "") not in feat_filter:
            continue
        target_evals.append(e)

    if not target_evals:
        print(
            f"ERROR: No evaluators matched. Available: {[e.name for e in all_evals]}",
            flush=True,
        )
        raise typer.Exit(code=1)
    print(f"  Matched: {[e.name for e in target_evals]}", flush=True)

    out_path = Path(output)
    out_path.mkdir(parents=True, exist_ok=True)
    all_sample_ids = list(labels_df["SAMPLE_ID"].unique())

    effective_chunk = _calculate_dynamic_chunk_size(chunk_size, len(all_sample_ids))
    print(f"  Effective chunk size: {effective_chunk}", flush=True)

    # ── Step 3: Extract each evaluator ──
    extracted = 0
    for e in target_evals:
        print(f"\n{'='*60}", flush=True)
        print(f"Extracting evaluator: {e.name}", flush=True)

        merged = _extract_evaluator(
            e,
            paths,
            all_sample_ids,
            effective_chunk,
            out_path,
            labels_df,
        )
        if merged is not None:
            extracted += 1

    elapsed = time.time() - t0
    print(
        f"\n=== Extract complete: {extracted}/{len(target_evals)} evaluators "
        f"in {elapsed:.1f}s ===",
        flush=True,
    )
    log.info(
        "extract_complete",
        n_extracted=extracted,
        n_total=len(target_evals),
        elapsed_sec=round(elapsed, 1),
    )


In [ ]:
#| export
@app.command()
def fuse(
    output_dir: Path = typer.Option(
        ..., "--output-dir", help="Directory containing *_matrix.parquet files"
    ),
    min_evaluators: int = typer.Option(
        1,
        "--min-evaluators",
        help="Minimum number of evaluators a sample must appear in to be retained",
    ),
    output_name: str = typer.Option(
        "super_matrix.parquet",
        "--output-name",
        help="Filename for the fused super-matrix (written to --output-dir)",
    ),
):
    """Fuse per-evaluator matrices into a single super-matrix.

    Discovers all ``*_matrix.parquet`` files in ``--output-dir``, extracts
    their feature columns (prefixed with evaluator name), outer-joins on
    SAMPLE_ID, and writes ``super_matrix.parquet`` for downstream multimodal
    evaluation.
    """
    import time as _time

    from kreview.core import fuse_matrices

    print("=== kreview fuse ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --output-dir     : {output_dir}", flush=True)
    print(f"  --min-evaluators : {min_evaluators}", flush=True)
    print(f"  --output-name    : {output_name}", flush=True)
    print("", flush=True)

    if min_evaluators < 1:
        print("ERROR: --min-evaluators must be >= 1", flush=True)
        raise typer.Exit(code=1)

    t0 = _time.time()
    result = fuse_matrices(
        output_dir,
        min_evaluators=min_evaluators,
        output_name=output_name,
    )

    if result.empty:
        print(
            "ERROR: No matrices found or all samples filtered. Nothing to fuse.",
            flush=True,
        )
        raise typer.Exit(code=1)

    n_features = len([c for c in result.columns if "__" in c])
    elapsed = _time.time() - t0
    print(
        f"  Fused: {len(result)} samples x {n_features} features "
        f"from {result.get('n_evaluators', pd.Series()).max() or '?'} evaluators "
        f"in {elapsed:.1f}s",
        flush=True,
    )
    print(f"  Output: {output_dir / output_name}", flush=True)


In [ ]:
#| export
@app.command("eval-gpu")
def eval_gpu(
    super_matrix: Path = typer.Option(
        ..., "--super-matrix", help="Path to super_matrix.parquet"
    ),
    model_type: str = typer.Option(
        "xgboost",
        "--model-type",
        help="Model backend: xgboost (CPU), tabicl (GPU), tabpfn (GPU)",
    ),
    output: Path = typer.Option("output/", help="Output directory for results"),
    cv_folds: int = typer.Option(5, "--cv-folds", help="Cross-validation folds"),
    impute_strategy: str = typer.Option(
        "median", "--impute-strategy", help="Imputation: median, mean, zero"
    ),
):
    """Run multimodal evaluation on a fused super-matrix.

    Currently supports XGBoost (CPU). GPU backends (TabICLv2, Real-TabPFN)
    will be added in a future release.
    """
    import time as _time
    import json
    import numpy as np

    print("=== kreview eval-gpu ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --super-matrix    : {super_matrix}", flush=True)
    print(f"  --model-type      : {model_type}", flush=True)
    print(f"  --output          : {output}", flush=True)
    print(f"  --cv-folds        : {cv_folds}", flush=True)
    print(f"  --impute-strategy : {impute_strategy}", flush=True)
    print("", flush=True)

    valid_models = {"xgboost", "tabicl", "tabpfn"}
    if model_type not in valid_models:
        print(f"ERROR: --model-type must be one of {valid_models}", flush=True)
        raise typer.Exit(code=1)

    if model_type in {"tabicl", "tabpfn"}:
        print(
            f"ERROR: GPU backend '{model_type}' is not yet available. "
            f"Use --model-type xgboost for CPU evaluation.",
            flush=True,
        )
        log.warning(
            "eval_gpu_backend_unavailable",
            model_type=model_type,
            msg="GPU backends will be available in a future release",
        )
        raise typer.Exit(code=1)

    if not super_matrix.exists():
        print(f"ERROR: Super-matrix not found: {super_matrix}", flush=True)
        raise typer.Exit(code=1)

    t0 = _time.time()
    df = pd.read_parquet(super_matrix)
    print(f"  Loaded: {df.shape[0]} samples x {df.shape[1]} columns", flush=True)

    # Separate feature columns (prefixed with __) from metadata
    feature_cols = [c for c in df.columns if "__" in c]
    if not feature_cols:
        print(
            "ERROR: No feature columns found (expected evaluator__feature format)",
            flush=True,
        )
        raise typer.Exit(code=1)

    print(f"  Features: {len(feature_cols)} multimodal features", flush=True)

    # Build binary target
    model_mask = df["label"].isin(
        [
            "True ctDNA+",
            "Possible ctDNA+",
            "Healthy Normal",
            "Possible ctDNA-",
            "Possible ctDNA\u2212",
        ]
    )
    model_df = df[model_mask].copy()
    y = model_df["label"].isin(["True ctDNA+", "Possible ctDNA+"]).astype(int).values

    if len(model_df) < 20 or len(np.unique(y)) < 2:
        print(
            f"ERROR: Insufficient data for modeling "
            f"(n={len(model_df)}, classes={len(np.unique(y))})",
            flush=True,
        )
        raise typer.Exit(code=1)

    # Impute missing values
    from kreview.cli import _impute

    X_imputed = _impute(model_df[feature_cols], impute_strategy)

    # Drop zero-variance features
    nonconst = [c for c in feature_cols if X_imputed[c].std() > 0]
    n_dropped = len(feature_cols) - len(nonconst)
    if n_dropped > 0:
        print(f"  Dropped {n_dropped} zero-variance features", flush=True)
    feature_cols = nonconst

    if not feature_cols:
        print("ERROR: All features have zero variance after imputation", flush=True)
        raise typer.Exit(code=1)

    X = X_imputed[feature_cols].values
    c_types = model_df.get("CANCER_TYPE", None)
    if c_types is not None:
        c_types = c_types.values
    a_types = model_df.get("access_version", None)
    if a_types is not None:
        a_types = a_types.values

    from kreview.eval_engine import single_feature_model

    print(f"  Training {model_type} with {len(feature_cols)} features...", flush=True)
    model_res, lr_model, rf_model, xgb_model = single_feature_model(
        X,
        y,
        feature_names=feature_cols,
        cancer_types=c_types,
        assays=a_types,
        n_folds=cv_folds,
    )

    out_path = Path(output)
    out_path.mkdir(parents=True, exist_ok=True)
    model_out = out_path / f"super_matrix_{model_type}_results.json"

    if "error" not in model_res:
        model_res["super_matrix_path"] = str(super_matrix)
        model_res["model_type"] = model_type
        model_res["n_multimodal_features"] = len(feature_cols)
        model_res["top_features"] = feature_cols

    with open(model_out, "w") as f:
        json.dump(model_res, f, indent=2, default=str)

    def _fmt(v):
        return "N/A" if v is None else f"{v:.3f}"

    elapsed = _time.time() - t0
    print(
        f"  Results: AUC_LR={_fmt(model_res.get('auc_lr'))}, "
        f"AUC_RF={_fmt(model_res.get('auc_rf'))}, "
        f"AUC_XGB={_fmt(model_res.get('auc_xgb'))} "
        f"in {elapsed:.1f}s",
        flush=True,
    )
    print(f"  Output: {model_out}", flush=True)
    log.info(
        "eval_gpu_complete",
        model_type=model_type,
        n_features=len(feature_cols),
        auc_lr=model_res.get("auc_lr"),
        auc_rf=model_res.get("auc_rf"),
        output=str(model_out),
    )